# SW-9-CSharp-JSONLD — JSON-LD avec dotNetRDF (twin C#)

Ce notebook est le **jumeau C# (.NET Interactive)** du notebook Python `SW-9-Python-JSONLD.ipynb`. Il explore **JSON-LD** (JavaScript Object Notation for Linked Data, W3C Recommendation 2014/2020) via **dotNetRDF** (`VDS.RDF.*`), le moteur .NET de reference — equivalent de `rdflib` + `pyld` cote Python.

## Objectifs pedagogiques

1. **Comprendre `@context`** : le dictionnaire de traduction entre termes JSON courts et IRIs RDF.
2. **Manipuler `@id` et `@type`** : identifier et typer les ressources dans un graphe.
3. **Parser du JSON-LD** en graphe RDF via `JsonLdParser` (charge un `TripleStore`).
4. **Serialiser un graphe RDF en JSON-LD** via `JsonLdWriter` (`TripleStore` -> JSON-LD).
5. **Explorer le vocabulaire Schema.org** et faire un round-trip JSON-LD <-> Turtle.

## Pourquoi un twin C# ?

Le notebook Python original s'appuie sur **`rdflib`** + **`jsonld`** (`pyld`). Ce twin C# invoque le **vrai moteur dotNetRDF** (`VDS.RDF.Parsing.JsonLdParser`, `VDS.RDF.Writing.JsonLdWriter`) — pas de workaround. Subtilite API : JSON-LD etant un format de **dataset** (graphe nomme, mot-cle `@graph`), le parser/writer dotNetRDF opere sur un `TripleStore` (collection de graphes), pas un `Graph` unique comme les autres serialisations RDF. Value-add (EPIC #4956 Prong B) : les deux twins sont des compagnons cross-langage sur le **meme standard W3C JSON-LD 1.1**.

## Prerequis
.NET 9.0+ (kernel `csharp`). dotNetRDF 3.x (restaure via la directive nuget).


## Setup — chargement de dotNetRDF


In [1]:
// dotNetRDF 3.x : meta-package incluant JsonLdParser / JsonLdWriter
#r "nuget: dotNetRDF, 3.4.1"

using System;
using System.Collections.Generic;
using System.Linq;
using System.Text;
using System.IO;
using Newtonsoft.Json;
using Newtonsoft.Json.Linq;
using VDS.RDF;
using VDS.RDF.Parsing;
using VDS.RDF.Writing;
using StringWriter = System.IO.StringWriter;

static void Show(object o) { try { display(o); } catch { Console.WriteLine(o); } }
static void Show(string label, object o) => Show($"{label}: {o}");

Show("dotNetRDF", "3.4.1 charge — JsonLdParser (IStoreReader) / JsonLdWriter (IStoreWriter) disponibles.");


The below script needs to be able to find the current output cell; this is an easy method to get it.

Installed Packages dotNetRDF, 3.4.1

dotNetRDF: 3.4.1 charge — JsonLdParser (IStoreReader) / JsonLdWriter (IStoreWriter) disponibles.

## Helpers de conversion RDF <-> JSON-LD

On definit ici trois fonctions utilitaires qui encapsulent la subtilite API : JSON-LD se charge dans un `TripleStore` (dataset), pas un `Graph` unique. On extrait ensuite le graphe par defaut pour l'interroger comme un graphe RDF classique.


In [2]:
// --- Helpers : JSON-LD <-> TripleStore <-> Graph + Turtle ---
// Parse JSON-LD -> TripleStore -> on renvoie le graphe par defaut (dataset a un seul graphe)
static IGraph LoadJsonLd(string json)
{
    var store = new TripleStore();
    using (var reader = new StringReader(json))
        new JsonLdParser().Load(store, reader);
    return store.Graphs.Count > 0 ? store.Graphs.First() : new Graph();
}

// Serialise un graphe RDF -> JSON-LD (on l'emballe dans un TripleStore pour le writer)
static string SerializeToJsonLd(IGraph graph)
{
    var store = new TripleStore();
    store.Add(graph, false);
    var sw = new StringWriter();
    new JsonLdWriter().Save(store, sw);
    return sw.ToString();
}

// Serialise un graphe RDF -> Turtle (comparaison)
static string SerializeToTurtle(IGraph graph)
{
    var sw = new StringWriter();
    new CompressingTurtleWriter().Save(graph, sw);
    return sw.ToString();
}

Show("Helpers", "LoadJsonLd / SerializeToJsonLd / SerializeToTurtle definis.");


Helpers: LoadJsonLd / SerializeToJsonLd / SerializeToTurtle definis.

## Partie 1 — `@context` : le dictionnaire de traduction

JSON-LD = JSON + un **`@context`** qui mappe les termes courts (cles JSON) vers des **IRIs** RDF (identifiants globaux). Cela rend JSON compatible Linked Data sans alourdir la syntaxe.

Exemple : `"name"` -> `http://schema.org/name`, `"Person"` -> `http://schema.org/Person`.

On construit d'abord un `@context` comme un `JObject` (Newtonsoft.Json), puis on l'utilise dans un document JSON-LD compact.


In [3]:
// --- @context : dictionnaire terme -> IRI (Schema.org) ---
var context = new JObject
{
    ["@vocab"] = "http://schema.org/",
    ["name"] = "http://schema.org/name",
    ["email"] = "http://schema.org/email",
    ["url"] = new JObject { ["@id"] = "http://schema.org/url", ["@type"] = "@id" },
    ["Person"] = "http://schema.org/Person",
    ["knows"] = "http://schema.org/knows"
};
Show("@context (brut)", context.ToString(Formatting.Indented));

// Document JSON-LD compact utilisant ce contexte
var person = new JObject
{
    ["@context"] = context,
    ["@id"] = "http://example.org/alice",
    ["@type"] = "Person",
    ["name"] = "Alice Martin",
    ["email"] = "alice@example.org",
    ["url"] = "http://alice.example.org",
    ["knows"] = new JArray { "http://example.org/bob", "http://example.org/carol" }
};
Show("Document JSON-LD compact", person.ToString(Formatting.Indented));


@context (brut): {
  "@vocab": "http://schema.org/",
  "name": "http://schema.org/name",
  "email": "http://schema.org/email",
  "url": {
    "@id": "http://schema.org/url",
    "@type": "@id"
  },
  "Person": "http://schema.org/Person",
  "knows": "http://schema.org/knows"
}

Document JSON-LD compact: {
  "@context": {
    "@vocab": "http://schema.org/",
    "name": "http://schema.org/name",
    "email": "http://schema.org/email",
    "url": {
      "@id": "http://schema.org/url",
      "@type": "@id"
    },
    "Person": "http://schema.org/Person",
    "knows": "http://schema.org/knows"
  },
  "@id": "http://example.org/alice",
  "@type": "Person",
  "name": "Alice Martin",
  "email": "alice@example.org",
  "url": "http://alice.example.org",
  "knows": [
    "http://example.org/bob",
    "http://example.org/carol"
  ]
}


warning CS1701: En supposant que la référence d'assembly 'System.Linq.Expressions, Version=6.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' utilisée par 'Newtonsoft.Json' correspond à l'identité 'System.Linq.Expressions, Version=10.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' de 'System.Linq.Expressions', il se peut que vous deviez fournir une stratégie runtime

warning CS1701: En supposant que la référence d'assembly 'System.Linq.Expressions, Version=6.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' utilisée par 'Newtonsoft.Json' correspond à l'identité 'System.Linq.Expressions, Version=10.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' de 'System.Linq.Expressions', il se peut que vous deviez fournir une stratégie runtime

warning CS1701: En supposant que la référence d'assembly 'System.Linq.Expressions, Version=6.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' utilisée par 'Newtonsoft.Json' correspond à l'identité 'System.Linq.Express

## Partie 2 — Parser du JSON-LD en graphe RDF

Le `JsonLdParser` de dotNetRDF lit un document JSON-LD dans un `TripleStore` (JSON-LD = format dataset). On extrait le graphe par defaut, puis on l'interroge comme n'importe quel graphe RDF — par exemple en le serialisant en Turtle.


In [4]:
// --- Parser JSON-LD -> TripleStore -> graphe par defaut (dotNetRDF JsonLdParser) ---
var jsonLdDoc = person.ToString(Formatting.Indented);
var g = LoadJsonLd(jsonLdDoc);
Show("Triplets RDF parses", g.Triples.Count);
Show("Graphe (Turtle)", SerializeToTurtle(g));

// On verifie les termes compacts -> IRIs complets : name doit pointer vers schema:name
var nameTriples = g.GetTriplesWithPredicate(g.CreateUriNode(new Uri("http://schema.org/name")));
Show("Valeur de schema:name", string.Join(", ", nameTriples.Select(t => t.Object.ToString())));


Triplets RDF parses: 6

Graphe (Turtle): @prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>.
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#>.
@prefix xsd: <http://www.w3.org/2001/XMLSchema#>.

<http://example.org/alice> <http://schema.org/email> "alice@example.org";
                           <http://schema.org/knows> "http://example.org/bob",
                                                     "http://example.org/carol";
                           <http://schema.org/name> "Alice Martin";
                           <http://schema.org/url> <http://alice.example.org/>;
                           a <http://schema.org/Person>.


Valeur de schema:name: Alice Martin^^http://www.w3.org/2001/XMLSchema#string


warning CS1701: En supposant que la référence d'assembly 'System.Linq.Expressions, Version=6.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' utilisée par 'Newtonsoft.Json' correspond à l'identité 'System.Linq.Expressions, Version=10.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' de 'System.Linq.Expressions', il se peut que vous deviez fournir une stratégie runtime



## Partie 3 — Serialiser un graphe RDF en JSON-LD

Inversement, le `JsonLdWriter` produit du JSON-LD (forme etendue) a partir d'un `TripleStore`. On reconstruit un graphe RDF a la main (assertion de triplets), on l'emballe dans un store, puis on le serialise.


In [5]:
// --- Construire un graphe RDF a la main, puis le serialiser en JSON-LD ---
var g2 = new Graph();
g2.NamespaceMap.AddNamespace("schema", new Uri("http://schema.org/"));
g2.NamespaceMap.AddNamespace("xsd", new Uri("http://www.w3.org/2001/XMLSchema#"));
var alice2 = g2.CreateUriNode("schema:alice");
var personType = g2.CreateUriNode("schema:Person");
var nameP = g2.CreateUriNode("schema:name");
var knowsP = g2.CreateUriNode("schema:knows");
var bob2 = g2.CreateUriNode("schema:bob");
g2.Assert(new Triple(alice2, g2.CreateUriNode("rdf:type"), personType));
g2.Assert(new Triple(alice2, nameP, g2.CreateLiteralNode("Alice", new Uri("http://www.w3.org/2001/XMLSchema#string"))));
g2.Assert(new Triple(alice2, knowsP, bob2));

Show("Graphe -> JSON-LD (forme etendue)", SerializeToJsonLd(g2));
Show("Graphe -> Turtle (comparaison)", SerializeToTurtle(g2));


Graphe -> JSON-LD (forme etendue): [
  {
    "@id": "http://schema.org/alice",
    "@type": [
      "http://schema.org/Person"
    ],
    "http://schema.org/name": [
      {
        "@value": "Alice"
      }
    ],
    "http://schema.org/knows": [
      {
        "@id": "http://schema.org/bob"
      }
    ]
  }
]

Graphe -> Turtle (comparaison): @prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>.
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#>.
@prefix xsd: <http://www.w3.org/2001/XMLSchema#>.
@prefix schema: <http://schema.org/>.

schema:alice schema:knows schema:bob;
             schema:name "Alice";
             a schema:Person.


## Partie 4 — Explorer le vocabulaire Schema.org

**Schema.org** est un vocabulaire collaboratif (Google, Microsoft, Yahoo, Yandex) pour decrire les choses du web : `Person`, `Product`, `Event`, `Place`, `Organization`... JSON-LD est le format recommande pour embarquer ces donnees structurees dans les pages HTML (SEO, rich snippets, knowledge graph).

On construit un document Schema.org `Product` (avec une offre imbriquee et une note agregee) et on le valide en le parsant.


In [6]:
// --- Document Schema.org Product en JSON-LD ---
var product = new JObject
{
    ["@context"] = new JObject { ["@vocab"] = "http://schema.org/" },
    ["@type"] = "Product",
    ["@id"] = "http://shop.example.org/produit/42",
    ["name"] = "Casque audio ANC",
    ["description"] = "Casque sans fil a reduction de bruit active",
    ["brand"] = new JObject { ["@type"] = "Brand", ["name"] = "AudioCo" },
    ["offers"] = new JObject
    {
        ["@type"] = "Offer",
        ["price"] = "199.99",
        ["priceCurrency"] = "EUR",
        ["availability"] = "http://schema.org/InStock"
    },
    ["aggregateRating"] = new JObject
    {
        ["@type"] = "AggregateRating",
        ["ratingValue"] = "4.5",
        ["reviewCount"] = "127"
    }
};
Show("Schema.org Product (JSON-LD)", product.ToString(Formatting.Indented));

// Validation : on parse et on compte les triplets generes
var g3 = LoadJsonLd(product.ToString(Formatting.Indented));
Show("Triplets generes (Product)", g3.Triples.Count);
// Les types distincts presents dans le graphe
var types = g3.Triples
    .Where(t => t.Predicate.ToString().EndsWith("type") || t.Predicate.ToString().EndsWith("#type"))
    .Select(t => t.Object.ToString())
    .Distinct();
Show("Types Schema.org rencontres", string.Join(", ", types));
// Le prix et la devise
var priceTriples = g3.GetTriplesWithPredicate(g3.CreateUriNode(new Uri("http://schema.org/price")));
Show("Offre (price)", string.Join(", ", priceTriples.Select(t => t.Object.ToString())));


Schema.org Product (JSON-LD): {
  "@context": {
    "@vocab": "http://schema.org/"
  },
  "@type": "Product",
  "@id": "http://shop.example.org/produit/42",
  "name": "Casque audio ANC",
  "description": "Casque sans fil a reduction de bruit active",
  "brand": {
    "@type": "Brand",
    "name": "AudioCo"
  },
  "offers": {
    "@type": "Offer",
    "price": "199.99",
    "priceCurrency": "EUR",
    "availability": "http://schema.org/InStock"
  },
  "aggregateRating": {
    "@type": "AggregateRating",
    "ratingValue": "4.5",
    "reviewCount": "127"
  }
}

Triplets generes (Product): 15

Types Schema.org rencontres: http://schema.org/Product, http://schema.org/AggregateRating, http://schema.org/Brand, http://schema.org/Offer

Offre (price): 199.99^^http://www.w3.org/2001/XMLSchema#string


warning CS1701: En supposant que la référence d'assembly 'System.Linq.Expressions, Version=6.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' utilisée par 'Newtonsoft.Json' correspond à l'identité 'System.Linq.Expressions, Version=10.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' de 'System.Linq.Expressions', il se peut que vous deviez fournir une stratégie runtime

warning CS1701: En supposant que la référence d'assembly 'System.Linq.Expressions, Version=6.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' utilisée par 'Newtonsoft.Json' correspond à l'identité 'System.Linq.Expressions, Version=10.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' de 'System.Linq.Expressions', il se peut que vous deviez fournir une stratégie runtime

warning CS1701: En supposant que la référence d'assembly 'System.Linq.Expressions, Version=6.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' utilisée par 'Newtonsoft.Json' correspond à l'identité 'System.Linq.Express

## Partie 5 — `@graph` : regrouper plusieurs ressources

Le mot-cle `@graph` permet de regrouper plusieurs ressources sous un meme `@context`, sans que le nœud racine ne soit lui-meme une ressource nommee. C'est la forme naturelle pour les datasets (plusieurs sujets decrits ensemble).


In [7]:
// --- @graph : dataset de plusieurs ressources ---
var dataset = new JObject
{
    ["@context"] = new JObject { ["@vocab"] = "http://schema.org/" },
    ["@graph"] = new JArray
    {
        new JObject { ["@id"] = "http://example.org/alice", ["@type"] = "Person", ["name"] = "Alice", ["knows"] = new JObject { ["@id"] = "http://example.org/bob" } },
        new JObject { ["@id"] = "http://example.org/bob", ["@type"] = "Person", ["name"] = "Bob" },
        new JObject { ["@id"] = "http://example.org/carol", ["@type"] = "Person", ["name"] = "Carol" }
    }
};
Show("Dataset @graph (3 personnes)", dataset.ToString(Formatting.Indented));

var g4 = LoadJsonLd(dataset.ToString(Formatting.Indented));
Show("Triplets (dataset)", g4.Triples.Count);
// Lien knows : Alice -> Bob
var aliceNode = g4.GetUriNode(new Uri("http://example.org/alice"));
var knowsPred = g4.GetUriNode(new Uri("http://schema.org/knows"));
var knowsTriples = g4.GetTriplesWithSubjectPredicate(aliceNode, knowsPred);
Show("Alice knows", string.Join(", ", knowsTriples.Select(t => t.Object.ToString())));


Dataset @graph (3 personnes): {
  "@context": {
    "@vocab": "http://schema.org/"
  },
  "@graph": [
    {
      "@id": "http://example.org/alice",
      "@type": "Person",
      "name": "Alice",
      "knows": {
        "@id": "http://example.org/bob"
      }
    },
    {
      "@id": "http://example.org/bob",
      "@type": "Person",
      "name": "Bob"
    },
    {
      "@id": "http://example.org/carol",
      "@type": "Person",
      "name": "Carol"
    }
  ]
}

Triplets (dataset): 7

Alice knows: http://example.org/bob


warning CS1701: En supposant que la référence d'assembly 'System.Linq.Expressions, Version=6.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' utilisée par 'Newtonsoft.Json' correspond à l'identité 'System.Linq.Expressions, Version=10.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' de 'System.Linq.Expressions', il se peut que vous deviez fournir une stratégie runtime

warning CS1701: En supposant que la référence d'assembly 'System.Linq.Expressions, Version=6.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' utilisée par 'Newtonsoft.Json' correspond à l'identité 'System.Linq.Expressions, Version=10.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' de 'System.Linq.Expressions', il se peut que vous deviez fournir une stratégie runtime

warning CS1701: En supposant que la référence d'assembly 'System.Linq.Expressions, Version=6.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' utilisée par 'Newtonsoft.Json' correspond à l'identité 'System.Linq.Express

## Partie 6 — Round-trip JSON-LD <-> Turtle

Un test classique de conformite : parser du JSON-LD, serialiser en Turtle, re-parser, et verifier qu'on retrouve le meme graphe (round-trip). On verifie l'isomorphisme par egalite de l'ensemble des triplets (sujet, predicat, objet).


In [8]:
// --- Round-trip JSON-LD -> Turtle -> JSON-LD ---
var original = new JObject
{
    ["@context"] = new JObject { ["@vocab"] = "http://schema.org/" },
    ["@id"] = "http://example.org/workshop",
    ["@type"] = "Event",
    ["name"] = "Atelier JSON-LD",
    ["startDate"] = "2026-07-15"
};
var g5 = LoadJsonLd(original.ToString(Formatting.Indented));
Show("Triplets originaux", g5.Triples.Count);

// Turtle intermediaire
var turtle5 = SerializeToTurtle(g5);
Show("Turtle intermediaire", turtle5);

// Re-parse du Turtle -> nouveau graphe
var g5b = new Graph();
new TurtleParser().Load(g5b, new StringReader(turtle5));
Show("Triplets apres round-trip", g5b.Triples.Count);

// Isomorphisme : memes triplets (S,P,O) dans les deux graphes ?
var same = g5.Triples.Count == g5b.Triples.Count
    && g5.Triples.All(t => g5b.Triples.Any(u => u.Subject.Equals(t.Subject) && u.Predicate.Equals(t.Predicate) && u.Object.Equals(t.Object)));
Show("Round-trip JSON-LD -> Turtle coherent ?", same ? "OUI (isomorphe)" : "NON");

// Re-serialisation JSON-LD pour fermer la boucle
Show("Re-serialisation JSON-LD (forme etendue)", SerializeToJsonLd(g5b));


Triplets originaux: 3

Turtle intermediaire: @prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>.
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#>.
@prefix xsd: <http://www.w3.org/2001/XMLSchema#>.

<http://example.org/workshop> <http://schema.org/name> "Atelier JSON-LD";
                              <http://schema.org/startDate> "2026-07-15";
                              a <http://schema.org/Event>.


Triplets apres round-trip: 3

Round-trip JSON-LD -> Turtle coherent ?: OUI (isomorphe)

Re-serialisation JSON-LD (forme etendue): [
  {
    "@id": "http://example.org/workshop",
    "http://schema.org/name": [
      {
        "@value": "Atelier JSON-LD"
      }
    ],
    "http://schema.org/startDate": [
      {
        "@value": "2026-07-15"
      }
    ],
    "@type": [
      "http://schema.org/Event"
    ]
  }
]


warning CS1701: En supposant que la référence d'assembly 'System.Linq.Expressions, Version=6.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' utilisée par 'Newtonsoft.Json' correspond à l'identité 'System.Linq.Expressions, Version=10.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' de 'System.Linq.Expressions', il se peut que vous deviez fournir une stratégie runtime

warning CS1701: En supposant que la référence d'assembly 'System.Linq.Expressions, Version=6.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' utilisée par 'Newtonsoft.Json' correspond à l'identité 'System.Linq.Expressions, Version=10.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' de 'System.Linq.Expressions', il se peut que vous deviez fournir une stratégie runtime

warning CS1701: En supposant que la référence d'assembly 'System.Linq.Expressions, Version=6.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' utilisée par 'Newtonsoft.Json' correspond à l'identité 'System.Linq.Express

## Conclusion

Ce twin C# a couvert les 6 piliers de JSON-LD avec dotNetRDF :

1. **`@context`** : dictionnaire terme -> IRI (Partie 1).
2. **Parsing JSON-LD -> graphe RDF** via `JsonLdParser` (Partie 2).
3. **Serialisation graphe -> JSON-LD** via `JsonLdWriter` (Partie 3).
4. **Schema.org** : vocabulaire de donnees structurees (Partie 4).
5. **`@graph`** : regroupement de ressources (Partie 5).
6. **Round-trip** JSON-LD <-> Turtle + isomorphisme (Partie 6).

Le notebook Python original utilise `rdflib` + `pyld` ; ce twin C# utilise **dotNetRDF** (`VDS.RDF.*`). Les deux sont des moteurs W3C complets. Value-add (EPIC #4956 Prong B) : comparaison cross-langage des API sur le meme standard JSON-LD 1.1. Subtilite API notee : `JsonLdParser`/`JsonLdWriter` operent sur `TripleStore` (dataset), refletant que JSON-LD est nativement un format multi-graphes (mot-cle `@graph`).

## Exercices


In [9]:
// Exercice 1 : Compaction avec un contexte personnalise
// TODO etudiant : ecrivez un contexte qui compacte un graphe de livres
// en utilisant les termes courts "titre", "auteur", "publie".
// Indice : definissez un JObject @context mappant chaque terme court a son IRI complet.
// Etape 1 : ecrire le contexte. Etape 2 : l'utiliser pour construire un document JSON-LD compact.
public static JObject BuildBiblioContext()
{
    // TODO etudiant : retournez un contexte du type {"@context": {"titre": "http://.../title", ...}}
    return new JObject();
}
Show("Contexte biblio (a completer)", BuildBiblioContext().ToString());


Contexte biblio (a completer): {}


warning CS1701: En supposant que la référence d'assembly 'System.Linq.Expressions, Version=6.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' utilisée par 'Newtonsoft.Json' correspond à l'identité 'System.Linq.Expressions, Version=10.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' de 'System.Linq.Expressions', il se peut que vous deviez fournir une stratégie runtime

warning CS1701: En supposant que la référence d'assembly 'System.Linq.Expressions, Version=6.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' utilisée par 'Newtonsoft.Json' correspond à l'identité 'System.Linq.Expressions, Version=10.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' de 'System.Linq.Expressions', il se peut que vous deviez fournir une stratégie runtime

warning CS1701: En supposant que la référence d'assembly 'System.Linq.Expressions, Version=6.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' utilisée par 'Newtonsoft.Json' correspond à l'identité 'System.Linq.Express

In [10]:
// Exercice 2 : Framing (extraire un sous-graphe type)
// TODO etudiant : implementez une selection qui extrait tous les nœuds
// de type schema:Person dans un graphe et retourne leur @id + nom.
// Indice : enumeratez g4.Triples, filtrez par predicat rdf:type == http://schema.org/Person.
// Etape 1 : trouver les sujets de type Person. Etape 2 : pour chacun, recuperer son schema:name.
public static List<JObject> ExtractPersons(IGraph g)
{
    // TODO etudiant : retournez la liste des {"@id":..., "name":...}
    return new List<JObject>();
}
var persons = ExtractPersons(g4);
Show("Personnes extraites (a completer)", persons.Count == 0 ? "0 (a completer)" : string.Join("; ", persons.Select(p => p.ToString())));


Personnes extraites (a completer): 0 (a completer)


warning CS1701: En supposant que la référence d'assembly 'System.Linq.Expressions, Version=6.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' utilisée par 'Newtonsoft.Json' correspond à l'identité 'System.Linq.Expressions, Version=10.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' de 'System.Linq.Expressions', il se peut que vous deviez fournir une stratégie runtime

warning CS1701: En supposant que la référence d'assembly 'System.Linq.Expressions, Version=6.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' utilisée par 'Newtonsoft.Json' correspond à l'identité 'System.Linq.Expressions, Version=10.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' de 'System.Linq.Expressions', il se peut que vous deviez fournir une stratégie runtime

warning CS1701: En supposant que la référence d'assembly 'System.Linq.Expressions, Version=6.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' utilisée par 'Newtonsoft.Json' correspond à l'identité 'System.Linq.Express

In [11]:
// Exercice 3 : Valider un JSON-LD mal forme
// TODO etudiant : ecrivez une fonction qui tente de parser un JSON-LD et
// retourne true s'il est valide (parsable en graphe RDF), false sinon.
// Indice : try/catch autour de LoadJsonLd (qui appelle JsonLdParser.Load).
public static bool IsValidJsonLd(string json)
{
    // TODO etudiant : retournez vrai si parsable, faux sinon
    return true;
}
Show("JSON-LD valide (cas correct)", IsValidJsonLd("{\"@id\":\"http://x.org/a\",\"@type\":\"http://x.org/T\",\"http://x.org/name\":\"foo\"}"));
Show("JSON-LD valide (cas casse)", IsValidJsonLd("{ce n'est pas du json}"));


JSON-LD valide (cas correct): True

JSON-LD valide (cas casse): True

---

*Twin C# du notebook Python `SW-9-Python-JSONLD.ipynb`. Marathon parite .NET <-> Python (#4956, Prong B). dotNetRDF 3.4.1 (vrai moteur W3C JSON-LD 1.1).*
